[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
import torch
import torch.nn as nn
import math

In [28]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = self.d_model//self.num_heads
        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)
        self.W_o = nn.Linear(d_model,d_model)

    def forward(self, x, cache=None):
        seq = x.shape[-2]
        q,k,v = self.W_q(x),self.W_k(x),self.W_v(x)
        q = q.view(-1,seq,self.num_heads,self.d_k).transpose(1,2).contiguous()
        k = k.view(-1,seq,self.num_heads,self.d_k).transpose(1,2).contiguous()
        v = v.view(-1,seq,self.num_heads,self.d_k).transpose(1,2).contiguous()
        if cache:
            cached_k,cached_v = cache
            k = torch.cat([cached_k,k],dim=-2)
            v = torch.cat([cached_v,v],dim=-2)
        score = torch.einsum('...sd,...kd->...sk',q,k)/math.sqrt(self.d_k)
        mask = torch.tril(torch.ones(seq,seq)).bool()
        score = score.masked_fill(~mask,float('-inf'))
        attn = torch.einsum('...sk,...kv->...sv',torch.softmax(score,dim=-1),v)
        attn = attn.transpose(1,2).contiguous().view(-1,seq,self.d_model)
        return self.W_o(attn),(k,v)

In [29]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)

inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [30]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (18.0ms)
  ✅ [2/5] Cache structure (6.7ms)
  ✅ [3/5] Decode step appends to cache (4.0ms)
  ✅ [4/5] Incremental decode matches full forward (5.5ms)
  ✅ [5/5] Gradient flow (4.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (38.3ms total)
  Progress saved. Run status() to see your dashboard.

